# Parameter and FLOPs

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
CLASSES = [ 
    'Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting', 'Cyclist', 'Tram', 'Misc'
]

NUM_CLASSES = len(CLASSES)

In [3]:
""" 
Information about architecture config:
"B" indicating a residual block
"S" is for scale prediction block
"U" is for upsampling the feature map
"""
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 4],
    (128, 3, 2),
    ["B", 6],
    (256, 3, 2),
    ["B", 8],
    (512, 3, 2),
    ["B", 8],
    (1024, 3, 2),
    ["B", 6],
    (512, 1, 1),
    (1024, 3, 1),

    # ONLY ONE DETECTION HEAD
    "S"
]

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.use_bn_act = bn_act

    def forward(self, x):
        if self.use_bn_act:
            return self.leaky(self.bn(self.conv(x)))
        else:
            return self.conv(x)


class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()
        self.layers = nn.ModuleList()
        for repeat in range(num_repeats):
            self.layers += [
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels, kernel_size=3, padding=1),
                )
            ]

        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x)
            else:
                x = layer(x)

        return x


class ScalePrediction(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            CNNBlock(
                2 * in_channels, 3 * (num_classes + 5), bn_act=False, kernel_size=1
            ),
        )
        
        self.num_classes = num_classes

    def forward(self, x):
        return (
            self.pred(x)
            .reshape(x.shape[0], 3, self.num_classes + 5, x.shape[2], x.shape[3])
            .permute(0, 1, 3, 4, 2)
        )

class SiStNet(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []
        route_connections = []
        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1)
                route_connections.pop()

        return outputs

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels
        
        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(
                    CNNBlock(
                        in_channels,
                        out_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=1 if kernel_size == 3 else 0,
                    )
                )
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(ResidualBlock(in_channels, num_repeats=num_repeats))

            elif isinstance(module, str):
                if module == "S":
                    layers += [
                        ResidualBlock(in_channels, use_residual=False, num_repeats=1),
                        CNNBlock(in_channels, in_channels // 2, kernel_size=1),
                        ScalePrediction(in_channels=in_channels // 2, num_classes=self.num_classes),
                    ]
                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2))
                    in_channels = in_channels * 3

        return layers

if __name__ == "__main__": 
    model = SiStNet(num_classes=NUM_CLASSES)

# Method 1

In [4]:
from thop import profile
from thop import clever_format

dummy = torch.randn(1, 3, 640, 640)

macs, params = profile(
    model,
    inputs=(dummy,),
    verbose=False
)

print(f"Params: {params:,}")
print(f"MACs : {macs:,}")
print(f"GMACs: {macs/1e9:.3f}")
print(f"FLOPs: {macs * 2 :,}")
print(f"GFLOPs: {macs * 2 /1e9:.1f}")

Params: 67,245,895.0
MACs : 83,725,926,400.0
GMACs: 83.726
FLOPs: 167,451,852,800.0
GFLOPs: 167.5


# Method 2

In [6]:
from ptflops import get_model_complexity_info

macs, params = get_model_complexity_info(
    model,
    (3, 640, 640),
    as_strings=False,
    print_per_layer_stat=False
)

print("MACs:", macs)
print(f"GMACs: {macs/1e9:.1f}")
print("FLOPs:", macs * 2)
print(f"FLOPs: {macs * 2/1e9:.1f}")
print("Params:", params)

MACs: 83596303600
GMACs: 83.6
FLOPs: 167192607200
FLOPs: 167.2
Params: 67245973


# Method 3

In [7]:
total = 0
for name, param in model.named_parameters():
    layer_params = param.numel()
    total += layer_params
    if layer_params > 100_000:  # Sadece büyük katmanları göster
        print(f"{name:60s} {layer_params/1e6:.3f} M")

layers.5.conv.weight                                         0.295 M
layers.6.layers.0.1.conv.weight                              0.295 M
layers.6.layers.1.1.conv.weight                              0.295 M
layers.6.layers.2.1.conv.weight                              0.295 M
layers.6.layers.3.1.conv.weight                              0.295 M
layers.6.layers.4.1.conv.weight                              0.295 M
layers.6.layers.5.1.conv.weight                              0.295 M
layers.6.layers.6.1.conv.weight                              0.295 M
layers.6.layers.7.1.conv.weight                              0.295 M
layers.7.conv.weight                                         1.180 M
layers.8.layers.0.0.conv.weight                              0.131 M
layers.8.layers.0.1.conv.weight                              1.180 M
layers.8.layers.1.0.conv.weight                              0.131 M
layers.8.layers.1.1.conv.weight                              1.180 M
layers.8.layers.2.0.conv.weight   

# F-RCNN

In [17]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def create_model(num_classes):
    
    # load Faster RCNN pre-trained model
    model = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(
        weights=None,          # COCO weights OFF
        weights_backbone=None  # ImageNet backbone OFF
    )
    
    # get the number of input features 
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # define a new head for the detector with required number of classes
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes) 

    return model

In [19]:
frnn_model = create_model(num_classes=NUM_CLASSES)

In [20]:
macs, params = get_model_complexity_info(
    frnn_model,
    (3, 640, 640),
    as_strings=True,
    print_per_layer_stat=True
)

print("MACs:", macs)
print("Params:", params)

FasterRCNN(
  18.99 M, 100.000% Params, 18.58 GMac, 99.776% MACs, 
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    4.44 M, 23.383% Params, 3.78 GMac, 20.294% MACs, 
    (body): IntermediateLayerGetter(
      2.97 M, 15.654% Params, 2.86 GMac, 15.370% MACs, 
      (0): Conv2dNormActivation(
        464, 0.002% Params, 74.24 MMac, 0.399% MACs, 
        (0): Conv2d(432, 0.002% Params, 69.12 MMac, 0.371% MACs, 3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, 0.000% Params, 5.12 MMac, 0.027% MACs, 16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): Hardswish(0, 0.000% Params, 0.0 Mac, 0.000% MACs, )
      )
      (1): InvertedResidual(
        464, 0.002% Params, 76.8 MMac, 0.412% MACs, 
        (block): Sequential(
          464, 0.002% Params, 76.8 

# EffDet

In [46]:
from effdet.config.model_config import efficientdet_model_param_dict
from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain
from effdet.efficientdet import HeadNet

MODEL_ARCHITECTURE='tf_efficientnetv2_s'

def effdet_model(num_classes=NUM_CLASSES, image_size=640, architecture=MODEL_ARCHITECTURE):
    efficientdet_model_param_dict[MODEL_ARCHITECTURE] = dict(
        name=MODEL_ARCHITECTURE,
        backbone_name=MODEL_ARCHITECTURE,
        backbone_args=dict(drop_path_rate=0.2),
        num_classes=num_classes,
        url='', )
    
    config = get_efficientdet_config(architecture)
    config.update({'num_classes': num_classes})
    config.update({'image_size': (image_size, image_size)})
    
    print(config)

    net = EfficientDet(config, pretrained_backbone=True)
    net.class_net = HeadNet(
        config,
        num_outputs=config.num_classes,
    )
    return DetBenchTrain(net, config)



In [47]:
model = effdet_model()

{'name': 'tf_efficientnetv2_s', 'backbone_name': 'tf_efficientnetv2_s', 'backbone_args': {'drop_path_rate': 0.2}, 'backbone_indices': None, 'image_size': [640, 640], 'num_classes': 8, 'min_level': 3, 'max_level': 7, 'num_levels': 5, 'num_scales': 3, 'aspect_ratios': [[1.0, 1.0], [1.4, 0.7], [0.7, 1.4]], 'anchor_scale': 4.0, 'pad_type': 'same', 'act_type': 'swish', 'norm_layer': None, 'norm_kwargs': {'eps': 0.001, 'momentum': 0.01}, 'box_class_repeats': 3, 'fpn_cell_repeats': 3, 'fpn_channels': 88, 'separable_conv': True, 'apply_resample_bn': True, 'conv_bn_relu_pattern': False, 'downsample_type': 'max', 'upsample_type': 'nearest', 'redundant_bias': True, 'head_bn_level_first': False, 'head_act_type': None, 'fpn_name': None, 'fpn_config': None, 'fpn_drop_path_rate': 0.0, 'alpha': 0.25, 'gamma': 1.5, 'label_smoothing': 0.0, 'legacy_focal': False, 'jit_loss': False, 'delta': 0.1, 'box_loss_weight': 50.0, 'soft_nms': False, 'max_detection_points': 5000, 'max_det_per_image': 100, 'url': ''}

In [48]:
macs, params = get_model_complexity_info(
    model.model,
    (3, 640, 640),
    as_strings=True,
    print_per_layer_stat=True
)

print("MACs:", macs)
print("Params:", params)

EfficientDet(
  19.96 M, 98.682% Params, 22.9 GMac, 99.569% MACs, 
  (backbone): EfficientNetFeatures(
    19.58 M, 96.792% Params, 21.96 GMac, 95.495% MACs, 
    (conv_stem): Conv2dSame(0, 0.000% Params, 0.0 Mac, 0.000% MACs, 3, 24, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn1): BatchNormAct2d(
      0, 0.000% Params, 0.0 Mac, 0.000% MACs, 24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity(0, 0.000% Params, 0.0 Mac, 0.000% MACs, )
      (act): SiLU(0, 0.000% Params, 0.0 Mac, 0.000% MACs, inplace=True)
    )
    (blocks): Sequential(
      19.58 M, 96.792% Params, 21.96 GMac, 95.495% MACs, 
      (0): Sequential(
        10.37 k, 0.051% Params, 1.06 GMac, 4.616% MACs, 
        (0): ConvBnAct(
          5.18 k, 0.026% Params, 530.84 MMac, 2.308% MACs, 
          (conv): Conv2d(5.18 k, 0.026% Params, 530.84 MMac, 2.308% MACs, 24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNormAct2d(
          

In [54]:
import torch
import sys
sys.path.append("../../../../YOLOv5")
from models.yolo import Model
from utils.general import check_yaml
import yaml

cfg = check_yaml('../../../../YOLOv5/models/yolov5m.yaml')

with open(cfg) as f:
    model_cfg = yaml.safe_load(f)

model = Model(model_cfg, ch=3, nc=NUM_CLASSES)

ModuleNotFoundError: No module named 'models'